We will start with "easy" predictive algorithm, mostly linear ones, that we will take from the sklearn library

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.metrics import make_scorer
from pathlib import Path

In [2]:
from utils import score, compute_costs

In [3]:
x_train = pd.read_csv(Path.cwd().parent / "data/german_credit_train.csv")
x_test = pd.read_csv(Path.cwd().parent / "data/german_credit_test.csv")

In [4]:
x_test.head()

,Id,CheckingStatus,LoanDuration,CreditHistory,LoanPurpose,LoanAmount,ExistingSavings,EmploymentDuration,InstallmentPercent,Sex,...,CurrentResidenceDuration,OwnsProperty,Age,InstallmentPlans,Housing,ExistingCreditsCount,Job,Dependents,Telephone,ForeignWorker
0,0,no_checking,9,prior_payments_delayed,car_new,1032,100_to_500,4_to_7,3,male,...,4,savings_insurance,41,none,own,1,management_self-employed,1,none,yes
1,1,less_0,5,all_credits_paid_back,car_new,1523,less_100,unemployed,2,female,...,2,real_estate,19,none,rent,1,management_self-employed,1,none,yes
2,2,no_checking,39,prior_payments_delayed,repairs,7150,500_to_1000,4_to_7,3,male,...,4,unknown,52,none,own,2,skilled,1,yes,yes
3,3,0_to_200,15,prior_payments_delayed,furniture,250,500_to_1000,4_to_7,3,male,...,2,savings_insurance,24,none,own,2,skilled,2,yes,yes
4,4,0_to_200,16,prior_payments_delayed,car_new,5551,100_to_500,1_to_4,3,male,...,3,car_other,34,none,rent,2,management_self-employed,1,none,yes


In [5]:
x_train.head()

,CheckingStatus,LoanDuration,CreditHistory,LoanPurpose,LoanAmount,ExistingSavings,EmploymentDuration,InstallmentPercent,Sex,OthersOnLoan,...,OwnsProperty,Age,InstallmentPlans,Housing,ExistingCreditsCount,Job,Dependents,Telephone,ForeignWorker,Risk
0,0_to_200,31,credits_paid_to_date,other,1889,100_to_500,less_1,3,female,none,...,savings_insurance,32,none,own,1,skilled,1,none,yes,No Risk
1,less_0,18,credits_paid_to_date,car_new,462,less_100,1_to_4,2,female,none,...,savings_insurance,37,stores,own,2,skilled,1,none,yes,No Risk
2,less_0,15,prior_payments_delayed,furniture,250,less_100,1_to_4,2,male,none,...,real_estate,28,none,own,2,skilled,1,yes,no,No Risk
3,0_to_200,28,credits_paid_to_date,retraining,3693,less_100,greater_7,3,male,none,...,savings_insurance,32,none,own,1,skilled,1,none,yes,No Risk
4,no_checking,28,prior_payments_delayed,education,6235,500_to_1000,greater_7,3,male,none,...,unknown,57,none,own,2,skilled,1,none,yes,Risk


In [6]:
x_test.drop(columns='Id')


,CheckingStatus,LoanDuration,CreditHistory,LoanPurpose,LoanAmount,ExistingSavings,EmploymentDuration,InstallmentPercent,Sex,OthersOnLoan,CurrentResidenceDuration,OwnsProperty,Age,InstallmentPlans,Housing,ExistingCreditsCount,Job,Dependents,Telephone,ForeignWorker
0,no_checking,9,prior_payments_delayed,car_new,1032,100_to_500,4_to_7,3,male,none,4,savings_insurance,41,none,own,1,management_self-employed,1,none,yes
1,less_0,5,all_credits_paid_back,car_new,1523,less_100,unemployed,2,female,none,2,real_estate,19,none,rent,1,management_self-employed,1,none,yes
2,no_checking,39,prior_payments_delayed,repairs,7150,500_to_1000,4_to_7,3,male,co-applicant,4,unknown,52,none,own,2,skilled,1,yes,yes
3,0_to_200,15,prior_payments_delayed,furniture,250,500_to_1000,4_to_7,3,male,none,2,savings_insurance,24,none,own,2,skilled,2,yes,yes
4,0_to_200,16,prior_payments_delayed,car_new,5551,100_to_500,1_to_4,3,male,none,3,car_other,34,none,rent,2,management_self-employed,1,none,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,no_checking,38,credits_paid_to_date,appliances,5308,100_to_500,4_to_7,3,male,none,3,savings_insurance,31,none,own,1,skilled,1,none,yes
997,less_0,31,credits_paid_to_date,retraining,1997,less_100,1_to_4,3,male,none,3,savings_insurance,31,none,own,2,skilled,2,none,yes
998,less_0,20,prior_payments_delayed,radio_tv,1155,greater_1000,1_to_4,3,male,none,3,savings_insurance,33,none,rent,2,skilled,1,yes,yes
999,less_0,4,credits_paid_to_date,car_new,250,less_100,unemployed,1,female,none,1,real_estate,23,none,rent,1,skilled,1,none,yes


In [11]:
y_train = x_train[['LoanAmount', 'Risk']]
x_train.drop(columns='Risk')

,CheckingStatus,LoanDuration,CreditHistory,LoanPurpose,LoanAmount,ExistingSavings,EmploymentDuration,InstallmentPercent,Sex,OthersOnLoan,CurrentResidenceDuration,OwnsProperty,Age,InstallmentPlans,Housing,ExistingCreditsCount,Job,Dependents,Telephone,ForeignWorker
0,0_to_200,31,credits_paid_to_date,other,1889,100_to_500,less_1,3,female,none,3,savings_insurance,32,none,own,1,skilled,1,none,yes
1,less_0,18,credits_paid_to_date,car_new,462,less_100,1_to_4,2,female,none,2,savings_insurance,37,stores,own,2,skilled,1,none,yes
2,less_0,15,prior_payments_delayed,furniture,250,less_100,1_to_4,2,male,none,3,real_estate,28,none,own,2,skilled,1,yes,no
3,0_to_200,28,credits_paid_to_date,retraining,3693,less_100,greater_7,3,male,none,2,savings_insurance,32,none,own,1,skilled,1,none,yes
4,no_checking,28,prior_payments_delayed,education,6235,500_to_1000,greater_7,3,male,none,3,unknown,57,none,own,2,skilled,1,none,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3994,greater_200,27,credits_paid_to_date,furniture,4650,less_100,1_to_4,3,male,none,4,savings_insurance,40,none,own,1,skilled,1,none,yes
3995,0_to_200,11,prior_payments_delayed,furniture,250,greater_1000,4_to_7,3,male,none,3,car_other,32,bank,own,1,unemployed,1,none,yes
3996,no_checking,32,outstanding_credit,appliances,6536,unknown,greater_7,5,male,co-applicant,5,unknown,54,stores,own,2,unskilled,2,yes,yes
3997,0_to_200,38,outstanding_credit,other,1597,500_to_1000,greater_7,3,female,co-applicant,3,savings_insurance,27,stores,own,1,skilled,1,none,yes


In [12]:
# As we use linear models, we want to deal only with numbers
numerical_columns = x_train.select_dtypes(include = np.number).columns.tolist()
categorical_columns = x_train.select_dtypes(exclude=np.number).columns.tolist()

In [13]:
x_valid, x_predictions, y_valid, y_true = train_test_split(x_train, y_train)

In [14]:
y_valid = y_valid.drop(columns = 'LoanAmount')

In [15]:
score_error = make_scorer(score)
# our score function is transformed to be used in our sklearn model

In [16]:
preprocessor = preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), numerical_columns),
        ("cat", OrdinalEncoder(), categorical_columns),
        ]
)

In [17]:
pipe = make_pipeline(preprocessor,
                     RidgeClassifier())



pipe.fit(x_valid, y_valid)

y_pred = pipe.predict(x_predictions)

score_error(y_true, y_pred, 'RidgeClassifier')



/Applications/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:1304: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


AttributeError: DataFrame has none of the following attributes: predict.